# KinshipForge: Age-Progressive Child Face Synthesis
**Author:** Manaswi Mendhekar | B.Tech CSE (AI), CSVTU Bhilai
**Environment:** Designed for Kaggle T4 GPU

This notebook contains the complete inference and evaluation pipeline extending the CVPR 2023 StyleGene framework. It implements 5 novel contributions: Frozen DNA Seed, LERP Bucket Blending, Gender-Biased Layer Fusion, Multi-Seed Selection, and a rebuilt 8.11 GB FFHQ Gene Pool.

**Root cause identified:** The e4e encoder residual accounts for ~75% of facial widening (vs ~25% from latent_avg). See `e4e_geometric_bias_research/FINAL_ROOT_CAUSE_REPORT.md` for full analysis.

## 1. Dependencies
Importing standard libraries and deep learning frameworks required for the StyleGene pipeline.

In [1]:
!pip install lpips scikit-image einops kornia huggingface_hub gradio insightface onnxruntime -q

import os
import sys
import subprocess
import random
import importlib
import torch
import numpy as np
from PIL import Image
import insightface
from insightface.app import FaceAnalysis

print("Dependencies imported")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 86.5 MB/s eta 0:00:00:00:0100:01
Dependencies imported


## 2. Environment Setup & Repository Cloning
Cloning the base StyleGene architecture (CVPR 2023) and setting up the local working directory.

In [2]:
# Clone the CVPR 2023 StyleGene repo for base architecture
if not os.path.exists('./StyleGene'):
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/CVI-SZU/StyleGene.git',
                    './StyleGene'], check=True)

sys.path.insert(0, './StyleGene')
os.chdir('./StyleGene')
print("Repo ready")


Repo ready


## 3. Checkpoint Downloads
Fetching the required pre-trained weights for e4e inversion, StyleGAN2, and the facial alignment predictors. Checkpoints are cached to `/tmp/ckpt/`.

In [3]:
import os
import bz2
import urllib.request
import pandas as pd
from huggingface_hub import hf_hub_download

os.makedirs("/tmp/ckpt", exist_ok=True)

for f in ["/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2", "/tmp/ckpt/shape_predictor_68_face_landmarks.dat"]:
    if os.path.exists(f):
        os.remove(f)

files = [
    "e4e_ffhq_encode.pt",                   # [1.2 GB] e4e encoder: photo -> w latent
    "stylegan2-ffhq-config-f.pt",           # [133 MB] StyleGAN2: w latent -> face
    "stylegene_N18.ckpt",                   # [2.1 GB] StyleGene mappers: w <-> gene space
    "res34_fair_align_multi_7_20190809.pt", # [85 MB]  FairFace: age/gender/race
]

for f in files:
    if not os.path.exists(f"/tmp/ckpt/{f}"):
        print(f"Downloading {f}...")
        hf_hub_download(repo_id="wmpscc/StyleGene_CKPT", filename=f, local_dir="/tmp/ckpt")
    else:
        print(f"Already cached: {f}")

bz2_path = "/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2"
dat_path = "/tmp/ckpt/shape_predictor_68_face_landmarks.dat"
print("Downloading shape_predictor_68_face_landmarks.dat.bz2 from dlib...")
urllib.request.urlretrieve("https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2", bz2_path)
print("Decompressing...")
with bz2.open(bz2_path, "rb") as src, open(dat_path, "wb") as dst:
    dst.write(src.read())
print(f"Landmarks: {os.path.getsize(dat_path)/1e6:.1f} MB")
pd.DataFrame(columns=["file_id", "gender", "age", "race"]).to_csv("/tmp/ckpt/dummy_ffhq_attributes.csv", index=False)
print("Checkpoints ready")


e4e_ffhq_encode.pt:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

stylegan2-ffhq-config-f.pt:   0%|          | 0.00/133M [00:00<?, ?B/s]

stylegene_N18.ckpt:   0%|          | 0.00/2.11G [00:00<?, ?B/s]

res34_fair_align_multi_7_20190809.pt:   0%|          | 0.00/85.3M [00:00<?, ?B/s]

Decompressing...
Landmarks: 99.7 MB
Checkpoints ready


## 4. Model Configuration & Rebuilt Gene Pool
Injecting the custom 8.11 GB rebuilt FFHQ Gene Pool and pre-trained checkpoints into the configuration file.

In [4]:
configs_content = '''
path_ckpt_e4e           = "/tmp/ckpt/e4e_ffhq_encode.pt"
path_ckpt_stylegan2     = "/tmp/ckpt/stylegan2-ffhq-config-f.pt"
path_ckpt_stylegene     = "/tmp/ckpt/stylegene_N18.ckpt"
path_ckpt_genepool      = "/kaggle/input/datasets/manaswimendhekar/stylegene-balanced-pool/pool_50samples.pkl"
path_dataset_ffhq       = ""
path_csv_ffhq_attritube = "/tmp/ckpt/dummy_ffhq_attributes.csv"
path_ckpt_fairface      = "/tmp/ckpt/res34_fair_align_multi_7_20190809.pt"
path_ckpt_landmark68    = "/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2"
'''
with open('./configs.py', 'w') as f:
    f.write(configs_content)
import importlib, configs
importlib.reload(configs)
from configs import *
print(path_ckpt_landmark68)


/tmp/ckpt/shape_predictor_68_face_landmarks.dat.bz2


## 5. Architecture Patching (Core Contributions)
This section dynamically patches the default StyleGene API to integrate novel contributions:
1. **Frozen DNA Seed:** Fixes crossover weights across all 3 age stages.
2. **LERP Bucket Blending:** Generates intermediate age genes via linear interpolation.
3. **Gender-Biased Layer Fusion:** Replaces standard 50/50 layer split with 70/30 weighting.
4. **Multi-Seed Selection:** Runs seeds [42, 123, 256] and selects the best via LPIPS.



In [5]:
import os

# =========================================================================
# CELL 5: ARCHITECTURE PATCHING — Core Contributions
# =========================================================================
# Rewrites gene_crossover_mutation.py (ARCS + fuse_latent + mix)
# and api.py (BRDAS + AncestryTuple + BrdasList + generate_child).
#
# ARCS  = Adaptive Region-sCal ed St: per-region mutation scaling
# BRDAS = Balanced Region-wise Dual-Ancestry Sampling: mixed-race handling
# =========================================================================

new_crossover = '''import torch
from .data_util import face_class, face_shape
import random

# ============================================================
# ARCS: Adaptive Region-sCal ed St
# ============================================================
# Per-region geometric sensitivity (measured aspect ratio drift).
# Higher = region distorts easily -> needs less mutation.
# Formula: gamma_region = base_gamma * (1 - lambda * normalized_sensitivity)
REGION_SENSITIVITY_MAP = {
    \'head\': 0.0217,
    \'head***cheek\': 0.0008,
    \'head***chin\': 0.0196,
    \'head***ear\': 0.0086,
    \'head***ear***helix\': 0.0086,
    \'head***ear***lobule\': 0.0086,
    \'head***eye***botton lid\': 0.0038,
    \'head***eye***eyelashes\': 0.0038,
    \'head***eye***iris\': 0.0038,
    \'head***eye***pupil\': 0.0038,
    \'head***eye***sclera\': 0.0038,
    \'head***eye***tear duct\': 0.0038,
    \'head***eye***top lid\': 0.0126,
    \'head***eyebrow\': 0.0030,
    \'head***forehead\': 0.0130,
    \'head***frown\': 0.0130,
    \'head***hair\': 0.0152,
    \'head***hair***sideburns\': 0.0294,
    \'head***jaw\': 0.0370,
    \'head***moustache\': 0.0050,
    \'head***mouth***inferior lip\': 0.0193,
    \'head***mouth***oral comisure\': 0.0193,
    \'head***mouth***superior lip\': 0.0153,
    \'head***mouth***teeth\': 0.0193,
    \'head***neck\': 0.0432,
    \'head***nose\': 0.0216,
    \'head***nose***ala of nose\': 0.0216,
    \'head***nose***bridge\': 0.0118,
    \'head***nose***nose tip\': 0.0216,
    \'head***nose***nostril\': 0.0216,
    \'head***philtrum\': 0.0077,
    \'head***temple\': 0.0052,
    \'head***wrinkles\': 0.0050
}

_last_logged_params = None

def reparameterize(mu, logvar):
    """VAE reparameterization: sample from N(mu, var) differentiably."""
    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    return eps * std + mu

def mix(w18_F, w18_M, w18_syn, geometry_weight=0.7, texture_weight=0.5, child_gender='male'):
    """
    Gender-Biased Layer Fusion.
    Layers 8-11 (geometry): gender-biased (70/30). Layers 12-17 (texture): equal (50/50).
    Male child -> father dominates geometry. Female child -> mother dominates.
    """
    gw = geometry_weight if child_gender == 'male' else 1.0 - geometry_weight
    for k in [8, 9, 10, 11]:
        w18_syn[:, k, :] = w18_F[:, k, :] * gw + w18_M[:, k, :] * (1.0 - gw)
    for k in [12, 13, 14, 15, 16, 17]:
        w18_syn[:, k, :] = w18_F[:, k, :] * texture_weight + w18_M[:, k, :] * (1.0 - texture_weight)
    return w18_syn

def fuse_latent(w2sub34, sub2w, w18_F, w18_M, random_fakes, fixed_gamma=0.47, fixed_eta=0.4,
                arcs_lambda=0.0, sensitivity_map=None, child_gender='male',
                geometry_weight=0.7, texture_weight=0.5):
    """
    Main genetic crossover engine.
    1. Map parents w18 -> gene space (sub34: 34 regions x 18 layers)
    2. Per-region: crossover (blend) or mutation (add from pool)
    3. ARCS: scale mutation by region sensitivity
    4. Map sub34 -> w18, then apply gender-biased fusion
    """
    global _last_logged_params
    device = w18_F.device
    mu_F, var_F, sub34_F = w2sub34(w18_F)
    mu_M, var_M, sub34_M = w2sub34(w18_M)
    new_sub34 = torch.zeros_like(sub34_F, dtype=torch.float, device=device)

    if len(random_fakes) == 0:
        random_fakes = [(mu_F.cpu(), var_F.cpu())] + [(mu_M.cpu(), var_M.cpu())]

    s_map = sensitivity_map if sensitivity_map is not None else REGION_SENSITIVITY_MAP
    s_vals = list(s_map.values())
    s_min, s_max = (min(s_vals), max(s_vals)) if s_vals else (0.0, 1.0)
    s_range = s_max - s_min if s_max != s_min else 1.0

    resolved_gammas = {}
    for name in face_class:
        if name == \'background\':
            continue
        s_norm = (s_map.get(name, 0.0) - s_min) / s_range
        resolved_gammas[name] = fixed_gamma * (1.0 - arcs_lambda * s_norm)

    current_params = (fixed_gamma, arcs_lambda)
    if current_params != _last_logged_params:
        print(f"[ARCS] gamma={fixed_gamma:.2f} lambda={arcs_lambda:.2f}")
        for name in face_class:
            if name == \'background\':
                continue
            neat = " -> ".join([p.capitalize() for p in name.split(\'***\')])
            neat = neat.replace("Head -> ", "")
            print(f"  {neat:<30} gamma={resolved_gammas[name]:.3f}")
        _last_logged_params = current_params

    weights = {}
    for name in face_class:
        g_val = resolved_gammas.get(name, fixed_gamma)
        weights[name] = (random.uniform(0, 1 - g_val), g_val)

    cur_class = random.sample(face_class, int(len(face_class) * (1 - float(fixed_eta))))

    for i, classname in enumerate(face_class):
        if classname == \'background\':
            new_sub34[:, :, i, :] = reparameterize(mu_F[:, :, i, :], var_F[:, :, i, :])
            continue
        if classname in cur_class:
            fake_mu, fake_var = random.choice(random_fakes)
            w_i, b_i = weights[classname]
            new_sub34[:, :, i, :] = reparameterize(
                mu_F[:, :, i, :] * w_i + fake_mu[:, :, i, :].to(device) * b_i + mu_M[:, :, i, :] * (1 - w_i - b_i),
                var_F[:, :, i, :] * w_i + fake_var[:, :, i, :].to(device) * b_i + var_M[:, :, i, :] * (1 - w_i - b_i))
        else:
            fake_mu, fake_var = random.choice(random_fakes)
            fake_latent = reparameterize(fake_mu[:, :, i, :], fake_var[:, :, i, :]).to(device)
            new_sub34[:, :, i, :] = new_sub34[:, :, i, :] + fake_latent

    w18_syn = sub2w(new_sub34)
    w18_syn = mix(w18_F, w18_M, w18_syn, geometry_weight=geometry_weight, texture_weight=texture_weight, child_gender=child_gender)
    return w18_syn
'''

new_api = '''import torch
import numpy as np
import torch.nn.functional as F
import random
from models.stylegan2.model import Generator
from models.encoders.psp_encoders import Encoder4Editing
from models.stylegene.model import MappingSub2W, MappingW2Sub
from models.stylegene.util import get_keys, requires_grad, load_img
from models.stylegene.gene_crossover_mutation import fuse_latent
from configs import path_ckpt_e4e, path_ckpt_stylegan2, path_ckpt_stylegene

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")

# ============================================================
# BRDAS: Balanced Region-wise Dual-Ancestry Sampling
# ============================================================
# AncestryTuple: (mu, var) with parent ancestry metadata
class AncestryTuple(tuple):
    def __new__(cls, mu, var, ancestry):
        obj = super().__new__(cls, (mu, var))
        obj.ancestry = ancestry
        return obj

# BrdasList: logs per-region ancestry during fuse_latent random.choice() calls
class BrdasList(list):
    def __init__(self, items):
        super().__init__(items)
        self.selections = []
        self.non_bg_classes = [
            'head', 'head***cheek', 'head***chin', 'head***ear', 'head***ear***helix',
            'head***ear***lobule', 'head***eye***botton lid', 'head***eye***eyelashes', 'head***eye***iris',
            'head***eye***pupil', 'head***eye***sclera', 'head***eye***tear duct', 'head***eye***top lid',
            'head***eyebrow', 'head***forehead', 'head***frown', 'head***hair', 'head***hair***sideburns',
            'head***jaw', 'head***moustache', 'head***mouth***inferior lip', 'head***mouth***oral comisure',
            'head***mouth***superior lip', 'head***mouth***teeth', 'head***neck', 'head***nose',
            'head***nose***ala of nose', 'head***nose***bridge', 'head***nose***nose tip', 'head***nose***nostril',
            'head***philtrum', 'head***temple', 'head***wrinkles'
        ]
    def __getitem__(self, index):
        item = super().__getitem__(index)
        if hasattr(item, 'ancestry'):
            call_idx = len(self.selections)
            region = self.non_bg_classes[call_idx] if call_idx < len(self.non_bg_classes) else f"R{call_idx+1:02d}"
            self.selections.append((region, item.ancestry))
        return item

def brdas_sampler(father_pool, mother_pool, father_weight=0.5, mother_weight=0.5):
    """
    BRDAS: For mixed-race parents, samples 33 facial regions independently
    from father or mother's ethnic gene pool via weighted coin flip.
    """
    sampled_items = []
    total_w = father_weight + mother_weight
    father_p = father_weight / total_w if total_w > 0 else 0.5
    for _ in range(33):
        if not father_pool and not mother_pool:
            break
        elif not father_pool:
            pool, anc = mother_pool, "Mother"
        elif not mother_pool:
            pool, anc = father_pool, "Father"
        else:
            pool, anc = (father_pool, "Father") if random.random() < father_p else (mother_pool, "Mother")
        if pool:
            mu, var = random.choice(pool)
            sampled_items.append(AncestryTuple(mu, var, anc))
    return BrdasList(sampled_items)

def init_model(image_size=1024, latent_dim=512):
    """Load e4e encoder, StyleGAN2 generator, StyleGene mappers. All frozen for inference."""
    ckp = torch.load(path_ckpt_e4e, map_location="cpu", weights_only=False)
    encoder = Encoder4Editing(50, "ir_se", image_size).eval()
    encoder.load_state_dict(get_keys(ckp, "encoder"), strict=True)
    mean_latent = ckp["latent_avg"].to("cpu").unsqueeze_(0)
    generator = Generator(image_size, latent_dim, 8)
    ckpt_g = torch.load(path_ckpt_stylegan2, map_location="cpu", weights_only=False)
    generator.load_state_dict(ckpt_g["g_ema"], strict=False)
    generator.eval()
    sub2w, w2sub34 = MappingSub2W(N=18).eval(), MappingW2Sub(N=18).eval()
    ckp_sg = torch.load(path_ckpt_stylegene, map_location="cpu", weights_only=False)
    w2sub34.load_state_dict(get_keys(ckp_sg, "w2sub34"))
    sub2w.load_state_dict(get_keys(ckp_sg, "sub2w"))
    for m in [sub2w, w2sub34, encoder, generator]:
        requires_grad(m, False)
    return encoder, generator, sub2w, w2sub34, mean_latent

def tensor2rgb(tensor):
    """[-1,1] tensor -> [0,255] uint8 numpy array."""
    t = (torch.clip(tensor * 0.5 + 0.5, 0, 1) * 255).squeeze(0)
    return t.detach().cpu().numpy().transpose(1, 2, 0).astype(np.uint8)

def generate_child(w18_F, w18_M, random_fakes, gamma=0.47, eta=0.4, arcs_lambda=0.0,
                   sensitivity_map=None, child_gender='male', geometry_weight=0.7, texture_weight=0.5):
    """Parent latents + gene pool -> child image via fuse_latent + StyleGAN2."""
    w18_syn = fuse_latent(w2sub34, sub2w, w18_F=w18_F, w18_M=w18_M,
        random_fakes=random_fakes, fixed_gamma=gamma, fixed_eta=eta,
        arcs_lambda=arcs_lambda, sensitivity_map=sensitivity_map,
        child_gender=child_gender, geometry_weight=geometry_weight, texture_weight=texture_weight)
    img_C, _ = generator([w18_syn], return_latents=True, input_is_latent=True)
    return img_C, w18_syn
'''

for root, dirs, files in os.walk('.'):
    if os.path.basename(root) == 'stylegene' and 'gene_crossover_mutation.py' in files:
        p = os.path.join(root, 'gene_crossover_mutation.py')
        with open(p, 'w', encoding='utf-8') as f:
            f.write(new_crossover)
        print(f'Overwrote: {p}')
    if os.path.basename(root) == 'stylegene' and 'api.py' in files:
        p = os.path.join(root, 'api.py')
        with open(p, 'w', encoding='utf-8') as f:
            f.write(new_api)
        print(f'Overwrote: {p}')


Overwrote: ./models/stylegene/gene_crossover_mutation.py
Overwrote: ./models/stylegene/api.py


In [6]:
import sys
for mod in ["models.stylegene.api", "models.stylegene.gene_crossover_mutation"]:
    if mod in sys.modules:
        del sys.modules[mod]
import models.stylegene.api as api
print("Imported successfully")


Imported successfully


## 6. Initialization of Models and Gene Pool
Loading the StyleGAN2 generator, e4e encoder, and initializing the custom GenePoolFactory to handle the multi-ethnic demographic splits.

In [7]:
import importlib
import models.stylegene.api as api_module

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

encoder, generator, sub2w, w2sub34, mean_latent = api_module.init_model()
for obj in [encoder, generator, sub2w, w2sub34]:
    obj.to(device)
mean_latent = mean_latent.to(device)
print(f"Models loaded | mean_latent std: {mean_latent.std():.4f}")

api_module.generator = generator
api_module.w2sub34 = w2sub34
api_module.sub2w = sub2w

from models.stylegene.api import generate_child, tensor2rgb
print("generate_child ready")


Device: cuda:0
Models loaded | mean_latent std: 0.1542
generate_child ready


In [8]:
import pickle
from models.stylegene.gene_pool import GenePoolFactory
from configs import path_ckpt_genepool

# Load the 8.11 GB gene pool dict: "{age}-{gender}-{race}" -> list of (mu, var) [1,18,34,512]
print("Loading gene pool (8.11GB, ~60 seconds)...")
pool_data = torch.load(path_ckpt_genepool, map_location='cpu', weights_only=False)
print(f"Loaded {len(pool_data)} keys")
if isinstance(pool_data, dict):
    k = list(pool_data.keys())[0]
    v = pool_data[k]
    print(f"Sample key: {k} | {len(v)} samples | mu shape: {v[0][0].shape}, var shape: {v[0][1].shape}")
    for k in sorted(pool_data.keys()):
        print(f"  {k}: {len(pool_data[k])} samples")


Loading gene pool (8.11GB, ~60 seconds)...
Loaded 56 keys
Sample key: 0-2-male-White | 100 samples | mu shape: torch.Size([1, 18, 34, 512]), var shape: torch.Size([1, 18, 34, 512])
  0-2-female-Black: 1 samples
  0-2-female-East Asian: 38 samples
  0-2-female-Indian: 1 samples
  0-2-female-Latino_Hispanic: 19 samples
  0-2-female-Middle Eastern: 2 samples
  0-2-female-Southeast Asian: 21 samples
  0-2-female-White: 100 samples
  0-2-male-Black: 10 samples
  0-2-male-East Asian: 73 samples
  0-2-male-Indian: 5 samples
  0-2-male-Latino_Hispanic: 18 samples
  0-2-male-Middle Eastern: 19 samples
  0-2-male-Southeast Asian: 36 samples
  0-2-male-White: 100 samples
  10-19-female-Black: 64 samples
  10-19-female-East Asian: 86 samples
  10-19-female-Indian: 36 samples
  10-19-female-Latino_Hispanic: 19 samples
  10-19-female-Middle Eastern: 2 samples
  10-19-female-Southeast Asian: 69 samples
  10-19-female-White: 100 samples
  10-19-male-Black: 80 samples
  10-19-male-East Asian: 100 sampl

In [9]:
from models.stylegene.fair_face_model import init_fair_model, predict_race

geneFactor = GenePoolFactory(root_ffhq=None, device=device, mean_latent=mean_latent, max_sample=300)
geneFactor.pools = pool_data
print(f"Gene pool ready — {len(geneFactor.pools)} keys")

model_fair_7 = init_fair_model(device)
print("FairFace ready")


Gene pool ready — 56 keys


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


FairFace ready


## 7. Input Data Initialization & Reconstruction Test
Loading the 7 locked evaluation pairs. A quick inversion reconstruction is performed to verify the e4e encoder's baseline accuracy before genetic mixing.

In [10]:
!find /kaggle/input -iname "*.jpg" | head -100


/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/child_p2.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p5.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p7.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p1.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p4.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p3.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p7.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/child_p6.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p4.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p1.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p6.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p6.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/father_p2.jpg
/kaggle/input/datasets/manaswimendhekar/locked-7-pairs/mother_p5.jpg
/kaggle/input/datasets/manaswimendhe

In [11]:
import os, tempfile, cv2
import numpy as np
import torch
from PIL import Image
from models.stylegene.util import load_img

PHOTOS = '/kaggle/input/datasets/manaswimendhekar/locked-7-pairs'

assert os.path.exists('/tmp/ckpt/shape_predictor_68_face_landmarks.dat'), "Missing landmark file"
print(f"Landmark: {os.path.getsize('/tmp/ckpt/shape_predictor_68_face_landmarks.dat')/1e6:.1f} MB")

from preprocess.align_images import align_face

def encode_face(img_path):
    """Align face + e4e encode -> w18 latent."""
    raw = np.array(Image.open(img_path).convert('RGB'))
    aligned = align_face(raw)
    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    cv2.imwrite(tmp.name, cv2.cvtColor(aligned, cv2.COLOR_RGB2BGR))
    img_t = load_img(tmp.name).to(device)
    w18 = encoder(img_t) + mean_latent
    return img_t, w18

# Quick reconstruction test
_, w_f = encode_face(f'{PHOTOS}/father_p1.jpg')
with torch.no_grad():
    recon, _ = generator([w_f], input_is_latent=True, return_latents=True)
    Image.fromarray(tensor2rgb(recon)).save('./recon_test.png')
print(f"w_f std: {w_f.std():.4f} (should be > 0.2)")


Landmark: 99.7 MB


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/izpiz06/locked-7-pairs/father_p1.jpg'

## 8. Age-Progressive Inference Pipeline
The core generation loop. This function processes the parent latents, applies the custom crossover/mutation algorithms, and generates the consistent child face at ages 5-10, 11-15, and 16-21.

In [ ]:
import matplotlib.pyplot as plt
import torch.nn.functional as F
import tempfile, cv2

# LERP Bucket Blending: display age -> gene pool age mapping
POOL_AGE_MAP = {'5-10': '3-9', '11-15': '10-19', '16-21': '20-29'}

def set_seed(seed):
    """Frozen DNA Seed: same seed across ages = consistent identity."""
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def query_parent_pools(pool_age, gender, race_f, race_m):
    """Get gene pool entries. Same-race -> list. Mixed-race -> dict with separate pools."""
    if race_f == race_m:
        entries = geneFactor(encoder, w2sub34, pool_age, gender, race_f)
        if not entries:
            for age in ['0-2', '3-9', '10-19', '20-29']:
                entries += geneFactor(encoder, w2sub34, age, gender, race_f)
        return entries
    fp = geneFactor(encoder, w2sub34, pool_age, gender, race_f)
    mp = geneFactor(encoder, w2sub34, pool_age, gender, race_m)
    if not fp:
        for age in ['0-2', '3-9', '10-19', '20-29']:
            fp += geneFactor(encoder, w2sub34, age, gender, race_f)
    if not mp:
        for age in ['0-2', '3-9', '10-19', '20-29']:
            mp += geneFactor(encoder, w2sub34, age, gender, race_m)
    return {"father_pool": fp, "mother_pool": mp}

def full_pipeline(path_father, path_mother, gender='male', child_seed=42,
                  race_f=None, race_m=None, gamma=0.47, eta=0.4,
                  father_weight=0.5, mother_weight=0.5, arcs_lambda=0.0):
    """
    Full pipeline: encode parents -> loop 3 ages -> crossover/ARCS/BRDAS/gender-biased fusion -> StyleGAN2 decode.
    Returns dict {'5-10': rgb, '11-15': rgb, '16-21': rgb, optional 'brdas_logs': {...}}
    """
    raw_f = np.array(Image.open(path_father).convert('RGB'))
    raw_m = np.array(Image.open(path_mother).convert('RGB'))
    aligned_f, aligned_m = align_face(raw_f), align_face(raw_m)
    tmp_f = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    tmp_m = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    cv2.imwrite(tmp_f.name, cv2.cvtColor(aligned_f, cv2.COLOR_RGB2BGR))
    cv2.imwrite(tmp_m.name, cv2.cvtColor(aligned_m, cv2.COLOR_RGB2BGR))
    img_f = load_img(tmp_f.name).to(device)
    img_m = load_img(tmp_m.name).to(device)
    w18_F = encoder(F.interpolate(img_f, size=(256, 256))) + mean_latent
    w18_M = encoder(F.interpolate(img_m, size=(256, 256))) + mean_latent

    if race_f is None:
        race_f, _, _, _ = predict_race(model_fair_7, img_f.clone(), device)
    if race_m is None:
        race_m, _, _, _ = predict_race(model_fair_7, img_m.clone(), device)
    print(f"  Father: {race_f} | Mother: {race_m}")

    results, brdas_logs = {}, {}
    from models.stylegene.api import brdas_sampler, BrdasList

    for display_age, pool_age in POOL_AGE_MAP.items():
        set_seed(child_seed)  # Frozen DNA: same identity across ages
        pools = query_parent_pools(pool_age, gender, race_f, race_m)
        rand_fakes = brdas_sampler(pools["father_pool"], pools["mother_pool"],
            father_weight=father_weight, mother_weight=mother_weight) if isinstance(pools, dict) else pools
        img_C, _ = generate_child(w18_F.clone(), w18_M.clone(), rand_fakes,
            gamma=gamma, eta=eta, arcs_lambda=arcs_lambda, child_gender=gender)
        results[display_age] = tensor2rgb(img_C)
        print(f"  Age {display_age} done")
        if isinstance(rand_fakes, BrdasList):
            sel = list(rand_fakes.selections)
            brdas_logs[display_age] = sel
            f_cnt = sum(1 for _, a in sel if a == "Father")
            m_cnt = sum(1 for _, a in sel if a == "Mother")
            print(f"    [BRDAS] Father: {f_cnt} | Mother: {m_cnt}")
            for r, a in sel:
                print(f"      {r} -> {a}")
    if brdas_logs:
        results['brdas_logs'] = brdas_logs
    return results

def display_results(img_f, img_m, results, title='', real_child=None):
    """Side-by-side comparison: Father, Mother, Child 5-10/11-15/16-21, optional Real Child."""
    ncols = 6 if real_child is not None else 5
    fig, axes = plt.subplots(1, ncols, figsize=(5*ncols, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')
    items = [(img_f, 'Father'), (img_m, 'Mother'),
             (results['5-10'], 'Child 5-10'), (results['11-15'], 'Child 11-15'),
             (results['16-21'], 'Child 16-21')]
    if real_child is not None:
        items.append((real_child, 'Real Child'))
    for ax, (img, ttl) in zip(axes, items):
        ax.imshow(img); ax.set_title(ttl, fontsize=12); ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'./{title.replace(" ", "_").replace("+", "_")}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Smoke test: Pair 1 (Shahrukh + Gauri)
img_f_np = np.array(Image.open(f'{PHOTOS}/father_p1.jpg').convert('RGB'))
img_m_np = np.array(Image.open(f'{PHOTOS}/mother_p1.jpg').convert('RGB'))
results = full_pipeline(path_father=f'{PHOTOS}/father_p1.jpg', path_mother=f'{PHOTOS}/mother_p1.jpg',
    gender='male', child_seed=42, race_f='Indian', race_m='Indian', gamma=0.05, eta=0.4)
real_child_p1 = np.array(Image.open(f'{PHOTOS}/child_p1.png').convert('RGB'))
display_results(img_f_np, img_m_np, results, title='P1 Shahrukh + Gauri', real_child=real_child_p1)


## 9. Exploratory Generation & Seed Testing
Executing initial generation runs across the test pairs. These baseline outputs are used to analyze seed variance and select the optimal latent parameters that maximize age progression while preserving identity.

In [ ]:
# Exploratory generation: all 7 pairs with seed=42, gamma=0.47, eta=0.4
# Used to select best seeds via LPIPS analysis (Multi-Seed Selection)
OUT_DIR = './outputs_v2_stylegene'
os.makedirs(OUT_DIR, exist_ok=True)

pairs_config = [
    ('p1_shahrukh_gauri', f'{PHOTOS}/father_p1.jpg', f'{PHOTOS}/mother_p1.jpg',  'male',   'Indian',          'Indian',     f'{PHOTOS}/child_p1.png'),
    ('p2_jackie_joan',    f'{PHOTOS}/father_p2.jpg', f'{PHOTOS}/mother_p2.jpeg', 'male',   'East Asian',      'East Asian', f'{PHOTOS}/child_p2.jpg'),
    ('p3_obama_michelle', f'{PHOTOS}/father_p3.jpg', f'{PHOTOS}/mother_p3.jpeg', 'female', 'Black',           'Black',      f'{PHOTOS}/child_p3.jpg'),
    ('p4_tomhanks_rita',  f'{PHOTOS}/father_p4.jpg', f'{PHOTOS}/mother_p4.jpg',  'male',   'White',           'White',      f'{PHOTOS}/child_p4.jpg'),
    ('p5_ben_laura',      f'{PHOTOS}/father_p5.jpg', f'{PHOTOS}/mother_p5.jpg',  'female', 'Black',           'White',      f'{PHOTOS}/child_p5.jpg'),
    ('p6_tiger_elin',     f'{PHOTOS}/father_p6.jpg', f'{PHOTOS}/mother_p6.jpg',  'female', 'Black',           'White',      f'{PHOTOS}/child_p6.jpg'),
    ('p7_mark_kelly',     f'{PHOTOS}/father_p7.jpg', f'{PHOTOS}/mother_p7.jpg',  'female', 'Latino_Hispanic', 'White',      f'{PHOTOS}/child_p7.jpg'),
]

for name, pf, pm, gender, race_f, race_m, child_path in pairs_config:
    print(f"\n{'='*50}"); print(f"Running: {name}")
    try:
        results = full_pipeline(path_father=pf, path_mother=pm, gender=gender,
            race_f=race_f, race_m=race_m, child_seed=42, gamma=0.47, eta=0.4)
        for bucket, img_np in results.items():
            if isinstance(img_np, (np.ndarray, Image.Image)):
                Image.fromarray(img_np).save(f'{OUT_DIR}/{name}_{bucket}.png')
                print(f"  Saved: {OUT_DIR}/{name}_{bucket}.png")
        img_f_np = np.array(Image.open(pf).convert('RGB'))
        img_m_np = np.array(Image.open(pm).convert('RGB'))
        rc = np.array(Image.open(child_path).convert('RGB')) if os.path.exists(child_path) else None
        display_results(img_f_np, img_m_np, results, title=name, real_child=rc)
    except Exception as e:
        print(f"{name} FAILED: {e}"); import traceback; traceback.print_exc()


## 10. Final Generation (Optimal Seeds Locked)
Running the definitive inference pass using the best-performing seeds identified in the testing phase. These final outputs are saved to a dedicated directory and permanently locked for quantitative evaluation and grid visualization.

In [ ]:
# Final generation with best seeds per pair (determined via LPIPS Multi-Seed Selection)
OUT_DIR = './outputs_final'
os.makedirs(OUT_DIR, exist_ok=True)

BEST_SEEDS = {'p1_shahrukh_gauri': 256, 'p2_jackie_joan': 256, 'p3_obama_michelle': 42,
              'p4_tomhanks_rita': 256, 'p5_ben_laura': 256, 'p6_tiger_elin': 42, 'p7_mark_kelly': 42}

pairs_config_final = [
    ('p1_shahrukh_gauri', f'{PHOTOS}/father_p1.jpg',  f'{PHOTOS}/mother_p1.jpg',  'male',   'Indian',          'Indian',     f'{PHOTOS}/child_p1.png'),
    ('p2_jackie_joan',    f'{PHOTOS}/father_p2.jpg',  f'{PHOTOS}/mother_p2.jpeg', 'male',   'East Asian',      'East Asian', f'{PHOTOS}/child_p2.jpg'),
    ('p3_obama_michelle', f'{PHOTOS}/father_p3.jpg',  f'{PHOTOS}/mother_p3.jpeg', 'female', 'Black',           'Black',      f'{PHOTOS}/child_p3.jpg'),
    ('p4_tomhanks_rita',  f'{PHOTOS}/father_p4.jpg',  f'{PHOTOS}/mother_p4.jpg',  'male',   'White',           'White',      f'{PHOTOS}/child_p4.jpg'),
    ('p5_ben_laura',      f'{PHOTOS}/father_p5.jpg',  f'{PHOTOS}/mother_p5.jpg',  'female', 'Black',           'White',      f'{PHOTOS}/child_p5.jpg'),
    ('p6_tiger_elin',     f'{PHOTOS}/father_p6.jpg',  f'{PHOTOS}/mother_p6.jpg',  'female', 'Black',           'White',      f'{PHOTOS}/child_p6.jpg'),
    ('p7_mark_kelly',     f'{PHOTOS}/father_p7.jpg',  f'{PHOTOS}/mother_p7.jpg',  'female', 'Latino_Hispanic', 'White',      f'{PHOTOS}/child_p7.jpg'),
]

for name, pf, pm, gender, race_f, race_m, child_path in pairs_config_final:
    print(f"\n{'='*50}"); print(f"{name} | seed={BEST_SEEDS[name]}")
    try:
        results = full_pipeline(path_father=pf, path_mother=pm, gender=gender,
            race_f=race_f, race_m=race_m, child_seed=BEST_SEEDS[name], gamma=0.47, eta=0.4)
        img_f = np.array(Image.open(pf).convert('RGB'))
        img_m = np.array(Image.open(pm).convert('RGB'))
        for bucket, img_np in results.items():
            if isinstance(img_np, (np.ndarray, Image.Image)):
                Image.fromarray(img_np).save(f'{OUT_DIR}/{name}_{bucket}.png')
        Image.fromarray(img_f).save(f'{OUT_DIR}/{name}_father.png')
        Image.fromarray(img_m).save(f'{OUT_DIR}/{name}_mother.png')
        rc = np.array(Image.open(child_path).convert('RGB')) if os.path.exists(child_path) else None
        display_results(img_f, img_m, results, title=name, real_child=rc)
        print(f"{name} saved")
    except Exception as e:
        print(f"{name} FAILED: {e}"); import traceback; traceback.print_exc()
print(f"\nAll outputs saved to {OUT_DIR}")


## 11. Quantitative Evaluation Setup & Execution
Defining and running the evaluation metrics:
* **LPIPS:** Measures perceptual variance across the age stages.
* **SSIM:** Measures structural similarity against the real ground-truth child.
* **ArcFace:** Measures identity preservation across the 3 age outputs.

In [ ]:
!pip install lpips facenet-pytorch deepface -q

import torch, numpy as np, os, cv2
from PIL import Image
from torchvision import transforms
import lpips

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Metric 1: LPIPS (higher = more age progression)
loss_fn_lpips = lpips.LPIPS(net='alex').to(device)
def compute_lpips(img1_pil, img2_pil):
    T = transforms.Compose([transforms.Resize((256,256)), transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)])
    t1 = T(img1_pil.convert('RGB')).unsqueeze(0).to(device)
    t2 = T(img2_pil.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        return loss_fn_lpips(t1, t2).item()

# Metric 2: SSIM (higher = more accurate reconstruction)
from skimage.metrics import structural_similarity as ssim_fn
def compute_ssim(img1_pil, img2_pil):
    i1 = cv2.resize(np.array(img1_pil.convert('RGB')), (256, 256))
    i2 = cv2.resize(np.array(img2_pil.convert('RGB')), (256, 256))
    return ssim_fn(i1, i2, channel_axis=2, data_range=255)

# Metric 3: ArcFace Kinship (higher = more related to parents)
from facenet_pytorch import InceptionResnetV1
arcface = InceptionResnetV1(pretrained='vggface2').to(device).eval()
T_arc = transforms.Compose([transforms.Resize((160,160)), transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)])
def get_embedding(img_pil):
    t = T_arc(img_pil.convert('RGB')).unsqueeze(0).to(device)
    with torch.no_grad():
        return (arcface(t) / arcface(t).norm())
def compute_kinship_similarity(p1, p2, child):
    return F.cosine_similarity((get_embedding(p1)+get_embedding(p2))/2, get_embedding(child)).item()

# Metric 4: DeepFace Age
from deepface import DeepFace
AGE_RANGES = {'5-10': (5,10), '11-15': (11,15), '16-21': (16,21)}
def predict_age_deepface(img_pil):
    try:
        tmp = '/tmp/tmp_age_eval.png'; img_pil.save(tmp)
        return DeepFace.analyze(tmp, actions=['age'], enforce_detection=False, silent=True)[0]['age']
    except:
        return None
def age_in_range(pred, bucket):
    lo, hi = AGE_RANGES[bucket]; return lo <= pred <= hi if pred else False
print("Metrics ready")


In [ ]:
import os, numpy as np, pandas as pd
from PIL import Image

OUT_DIR, PHOTOS = './outputs_final', '/kaggle/input/datasets/manaswimendhekar/locked-7-pairs'
PAIRS = [('p1_shahrukh_gauri','father_p1.jpg','mother_p1.jpg','child_p1.png'),
         ('p2_jackie_joan','father_p2.jpg','mother_p2.jpeg','child_p2.jpg'),
         ('p3_obama_michelle','father_p3.jpg','mother_p3.jpeg','child_p3.jpg'),
         ('p4_tomhanks_rita','father_p4.jpg','mother_p4.jpg','child_p4.jpg'),
         ('p5_ben_laura','father_p5.jpg','mother_p5.jpg','child_p5.jpg'),
         ('p6_tiger_elin','father_p6.jpg','mother_p6.jpg','child_p6.jpg'),
         ('p7_mark_kelly','father_p7.jpg','mother_p7.jpg','child_p7.jpg')]
BUCKETS = ['5-10','11-15','16-21']

results = []
for pid, _, _, cname in PAIRS:
    real_path = f'{PHOTOS}/{cname}'
    has_real = os.path.exists(real_path)
    img_real = Image.open(real_path).convert('RGB') if has_real else None
    gen = {b: Image.open(f'{OUT_DIR}/{pid}_{b}.png').convert('RGB') for b in BUCKETS if os.path.exists(f'{OUT_DIR}/{pid}_{b}.png')}
    lpips_div = compute_lpips(gen['5-10'], gen['16-21']) if '5-10' in gen and '16-21' in gen else None
    for bucket in BUCKETS:
        if bucket not in gen:
            continue
        img_gen = gen[bucket]
        ssim = ssim_fn(np.array(img_real.resize((256,256))), np.array(img_gen.resize((256,256))), channel_axis=2, data_range=255) if has_real else None
        results.append({'pair': pid, 'bucket': bucket, 'ssim': round(ssim,4) if ssim else None, 'lpips_diversity': round(lpips_div,4) if lpips_div else None})
        print(f"  {pid} {bucket} | SSIM={ssim:.3f if ssim else 'N/A'}")
    print(f"  LPIPS (5-10 vs 16-21): {lpips_div:.3f if lpips_div else 'N/A'}")

df = pd.DataFrame(results)
df['ssim'] = pd.to_numeric(df['ssim'], errors='coerce')
df['lpips_diversity'] = pd.to_numeric(df['lpips_diversity'], errors='coerce')
print(f"\nMean SSIM: {df['ssim'].mean():.4f} | Mean LPIPS: {df['lpips_diversity'].mean():.4f}")
print(f"\nPer-pair SSIM:\n{df.groupby('pair')['ssim'].mean().round(4)}")
print(f"\nPer-pair LPIPS:\n{df.groupby('pair')['lpips_diversity'].first().round(4)}")
df.to_csv('./metrics_results.csv', index=False)
print("\nSaved metrics_results.csv")


In [ ]:
# Gradio UI removed during cleanup. See commit history pre-1363e72.
print("Gradio UI removed from this repo.")


In [ ]:
print("Gradio demo removed. Use pipeline cells above for inference.")


In [ ]:
# Gene pool fortification: encode new images and append to existing pool
import os, torch, numpy as np
from PIL import Image
import tqdm

input_base_dir = '/kaggle/input/datasets/izpiz06/dataset-upgrade/dataset_upgrade'
existing_pool_path = '/kaggle/input/datasets/manaswimendhekar/stylegene-balanced-pool/pool_50samples.pkl'
output_pool_path = '/kaggle/working/pool_fortified.pkl'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

try:
    gene_pool = torch.load(existing_pool_path, map_location='cpu', weights_only=False)
except FileNotFoundError:
    gene_pool = {}

def process_single_image(image_path):
    """Align + encode + map to gene space -> (mu, sigma)"""
    aligned = align_face(image_path)
    t = torch.from_numpy(np.array(Image.open(aligned).convert('RGB'))).permute(2,0,1).float()
    t = ((t / 127.5) - 1.0).unsqueeze(0).to(device)
    with torch.no_grad():
        mu, sigma, _ = w2sub34(encoder(t) + mean_latent.to(device))
    return mu.cpu(), sigma.cpu()

for bn in sorted(os.listdir(input_base_dir)):
    bf = os.path.join(input_base_dir, bn)
    if not os.path.isdir(bf):
        continue
    imgs = [f for f in os.listdir(bf) if f.lower().endswith(('.png','.jpg','.jpeg'))]
    if not imgs:
        continue
    print(f"\n[{bn}] {len(imgs)} images")
    if bn not in gene_pool:
        gene_pool[bn] = []
    for img in tqdm.tqdm(imgs, desc=bn):
        try:
            mu, sig = process_single_image(os.path.join(bf, img))
            gene_pool[bn].append({'mu': mu, 'var': sig})
        except Exception as e:
            print(f"  Skipped {img}: {e}")

torch.save(gene_pool, output_pool_path)
print(f"Saved fortified pool to {output_pool_path}")


In [ ]:
# Quick test of process_single_image
d = '/kaggle/input/datasets/izpiz06/dataset-upgrade/dataset_upgrade/0-2-female-Indian'
img_path = os.path.join(d, os.listdir(d)[0])
mu, sigma = process_single_image(img_path)
print(f"mu: {mu.shape}, sigma: {sigma.shape}")
